# 🚁 Low-Level DQN — Evaluation Notebook

Notebook này chạy model đã train và vẽ đầy đủ các biểu đồ đánh giá:

1. **Trajectory 3D** — đường bay thực tế vs đường thẳng lý tưởng  
2. **Distance to goal** theo từng step  
3. **Reward breakdown** (goal cosine / collision / battery)  
4. **Action distribution** — model đang ưu tiên action nào  
5. **Success rate & steps-to-reach** qua nhiều episode  
6. **Battery drain** theo thời gian  
7. **Q-value heatmap** theo không gian 2D (XY slice)  
8. **Multi-episode summary statistics**

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d.proj3d import proj_transform
from stable_baselines3 import DQN

from env_lowlevel import LowLevelStandaloneDQN, ACTION_DELTAS, REACH_THRESHOLD

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 1.8,
})

ACTION_NAMES = ['Lên (+z)', 'Xuống (-z)', 'Trái (+x)', 'Phải (-x)',
                'Tới (+y)', 'Lùi (-y)', 'Đứng im']
ACTION_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#95a5a6']

print('✅ Imports OK')

## 1. Load model & cấu hình

In [ ]:
# ── Đường dẫn tới model đã train ─────────────────────────────
MODEL_PATH = './models/lowlevel/dqn_final'   # thay đổi nếu cần
N_EVAL_EPISODES = 20   # số episode để thống kê
SEED = 42

# Load model
model = DQN.load(MODEL_PATH)
print(f'✅ Model loaded từ: {MODEL_PATH}')
print(f'   Policy network: {model.policy}')

# Tạo env test (không dùng VecEnv để dễ lấy info)
env = LowLevelStandaloneDQN(config={'num_drones': 3, 'gui': False})
print(f'   Obs space : {env.observation_space.shape}')  # (20,)
print(f'   Act space : {env.action_space.n}')           # 7

## 2. Hàm chạy 1 episode & thu thập dữ liệu

In [ ]:
def run_episode(env, model, seed=None, fixed_target=None, deterministic=True):
    """
    Chạy 1 episode, trả về dict chứa toàn bộ trajectory & metrics.
    """
    obs, info = env.reset(seed=seed)
    
    if fixed_target is not None:
        env.target_pos = np.array(fixed_target, dtype=np.float32)
        # Rebuild obs với target mới
        obs_matrix = env._get_obs_matrix()
        obs = env._build_obs(obs_matrix)

    target = env.target_pos.copy()

    # Buffers
    positions, velocities, batteries = [], [], []
    rewards, dists, actions, q_values_list = [], [], [], []

    done = False
    while not done:
        obs_matrix = env._get_obs_matrix()
        pos = obs_matrix[env.agent_idx, :3].copy()
        vel = obs_matrix[env.agent_idx, 3:6].copy()
        bat = float(env.battery[env.agent_idx])

        # Lấy Q-values thô (để vẽ heatmap sau)
        import torch
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(model.device)
        with torch.no_grad():
            q_vals = model.q_net(obs_tensor).cpu().numpy().flatten()

        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(int(action))
        done = terminated or truncated

        positions.append(pos)
        velocities.append(vel)
        batteries.append(bat)
        rewards.append(reward)
        dists.append(info['dist_to_goal'])
        actions.append(int(action))
        q_values_list.append(q_vals)

    return {
        'positions':  np.array(positions),
        'velocities': np.array(velocities),
        'batteries':  np.array(batteries),
        'rewards':    np.array(rewards),
        'dists':      np.array(dists),
        'actions':    np.array(actions),
        'q_values':   np.array(q_values_list),
        'target':     target,
        'reached':    info['reached'],
        'n_steps':    len(rewards),
        'total_reward': float(np.sum(rewards)),
    }

print('✅ Hàm run_episode sẵn sàng')

## 3. Chạy 1 episode demo — xem chi tiết

In [ ]:
ep = run_episode(env, model, seed=SEED)

print(f"Target     : {ep['target']}")
print(f"Đã tới đích: {'✅ CÓ' if ep['reached'] else '❌ KHÔNG'}")
print(f"Số steps   : {ep['n_steps']}")
print(f"Total reward: {ep['total_reward']:.2f}")
print(f"Dist cuối   : {ep['dists'][-1]:.3f} m")

## 4. Biểu đồ 1 — Trajectory 3D

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

pos  = ep['positions']
tgt  = ep['target']
start = pos[0]

# Màu theo thời gian: đậm → nhạt
n = len(pos)
colors = plt.cm.plasma(np.linspace(0.1, 0.9, n))

for i in range(n - 1):
    ax.plot(pos[i:i+2, 0], pos[i:i+2, 1], pos[i:i+2, 2],
            color=colors[i], alpha=0.85, linewidth=1.5)

# Đường thẳng lý tưởng
ax.plot([start[0], tgt[0]], [start[1], tgt[1]], [start[2], tgt[2]],
        'k--', alpha=0.35, linewidth=1.2, label='Đường thẳng lý tưởng')

# Điểm đặc biệt
ax.scatter(*start, s=120, color='lime', zorder=5, label='Xuất phát', edgecolors='k')
ax.scatter(*tgt,   s=200, marker='*', color='gold', zorder=5, label='Đích', edgecolors='k')
ax.scatter(*pos[-1], s=80, color='red', zorder=5, label='Kết thúc', edgecolors='k')

# Vùng reach threshold
u = np.linspace(0, 2*np.pi, 30)
v = np.linspace(0, np.pi, 20)
xs = REACH_THRESHOLD * np.outer(np.cos(u), np.sin(v)) + tgt[0]
ys = REACH_THRESHOLD * np.outer(np.sin(u), np.sin(v)) + tgt[1]
zs = REACH_THRESHOLD * np.outer(np.ones(np.size(u)), np.cos(v)) + tgt[2]
ax.plot_surface(xs, ys, zs, color='gold', alpha=0.12)

# Colorbar
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, n))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Step')

ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title(f"Trajectory 3D  |  {'Đến đích ✅' if ep['reached'] else 'Không đến đích ❌'}  |  {n} steps")
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('plot_trajectory3d.png', bbox_inches='tight')
plt.show()

## 5. Biểu đồ 2 — Distance, Reward, Battery, Velocity theo step

In [ ]:
steps = np.arange(len(ep['rewards']))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Metrics theo từng step — Episode demo', fontsize=14, fontweight='bold')

# ── (1) Distance to goal ──────────────────────────────────────
ax = axes[0, 0]
ax.plot(steps, ep['dists'], color='#e74c3c', linewidth=2)
ax.axhline(REACH_THRESHOLD, color='gold', linestyle='--', linewidth=1.5, label=f'Reach threshold ({REACH_THRESHOLD}m)')
ax.fill_between(steps, ep['dists'], REACH_THRESHOLD,
                where=np.array(ep['dists']) <= REACH_THRESHOLD,
                alpha=0.3, color='gold', label='Đã trong vùng đích')
ax.set_title('Distance to Goal (m)')
ax.set_xlabel('Step'); ax.set_ylabel('Khoảng cách (m)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── (2) Cumulative reward ─────────────────────────────────────
ax = axes[0, 1]
cum_reward = np.cumsum(ep['rewards'])
ax.plot(steps, ep['rewards'], color='#3498db', alpha=0.5, linewidth=1, label='Reward/step')
ax.plot(steps, cum_reward, color='#2980b9', linewidth=2, label='Cumulative reward')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_title('Reward theo step')
ax.set_xlabel('Step'); ax.set_ylabel('Reward')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── (3) Battery ───────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(steps, ep['batteries'], color='#27ae60', linewidth=2)
ax.axhline(0.15, color='orange', linestyle='--', linewidth=1.5, label='Low battery (15%)')
ax.fill_between(steps, ep['batteries'], 0.15,
                where=np.array(ep['batteries']) < 0.15,
                alpha=0.3, color='red', label='Vùng nguy hiểm')
ax.set_ylim(0, 1.05)
ax.set_title('Battery Level')
ax.set_xlabel('Step'); ax.set_ylabel('Pin (%)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── (4) Velocity magnitude ────────────────────────────────────
ax = axes[1, 1]
vel_mag = np.linalg.norm(ep['velocities'], axis=1)
ax.plot(steps, ep['velocities'][:, 0], color='#e74c3c', alpha=0.8, linewidth=1.2, label='vx')
ax.plot(steps, ep['velocities'][:, 1], color='#27ae60', alpha=0.8, linewidth=1.2, label='vy')
ax.plot(steps, ep['velocities'][:, 2], color='#3498db', alpha=0.8, linewidth=1.2, label='vz')
ax.plot(steps, vel_mag, color='k', linewidth=2, label='|v| (total)')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_title('Velocity (m/s)')
ax.set_xlabel('Step'); ax.set_ylabel('Tốc độ (m/s)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_step_metrics.png', bbox_inches='tight')
plt.show()

## 6. Biểu đồ 3 — Action distribution & Q-values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Action Analysis', fontsize=14, fontweight='bold')

# ── (1) Action frequency bar chart ───────────────────────────
ax = axes[0]
action_counts = np.bincount(ep['actions'], minlength=7)
bars = ax.bar(ACTION_NAMES, action_counts, color=ACTION_COLORS, edgecolor='white', linewidth=1.2)

# Thêm % trên mỗi bar
total_actions = len(ep['actions'])
for bar, count in zip(bars, action_counts):
    pct = count / total_actions * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title(f'Action Distribution ({total_actions} steps tổng)')
ax.set_ylabel('Số lần chọn')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

# ── (2) Q-values theo step (mean ± std) ──────────────────────
ax = axes[1]
q_vals = ep['q_values']    # (n_steps, 7)
steps_q = np.arange(len(q_vals))

for a_idx in range(7):
    ax.plot(steps_q, q_vals[:, a_idx],
            color=ACTION_COLORS[a_idx], alpha=0.75, linewidth=1.2,
            label=ACTION_NAMES[a_idx])

# Đánh dấu action thực sự được chọn
chosen_q = [q_vals[t, ep['actions'][t]] for t in range(len(ep['actions']))]
ax.scatter(steps_q, chosen_q, c='k', s=12, zorder=5, alpha=0.5, label='Action được chọn')

ax.set_title('Q-values theo step (mỗi đường = 1 action)')
ax.set_xlabel('Step'); ax.set_ylabel('Q-value')
ax.legend(fontsize=8, ncol=2, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_action_analysis.png', bbox_inches='tight')
plt.show()

## 7. Biểu đồ 4 — Q-value Heatmap (XY plane tại Z cố định)

In [ ]:
import torch

def compute_qmap(model, env, target, z_fixed=1.0, grid_size=25):
    """
    Tính max Q-value tại mỗi điểm (x, y) trên grid với z = z_fixed.
    Các thành phần obs khác (vel, bat, neighbors) giữ ở giá trị mặc định.
    """
    xs = np.linspace(0, 5, grid_size)
    ys = np.linspace(-2.5, 2.5, grid_size)
    
    Q_max = np.zeros((grid_size, grid_size))
    Q_best_action = np.zeros((grid_size, grid_size), dtype=int)
    
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            pos = np.array([x, y, z_fixed], dtype=np.float32)
            target_delta = target - pos
            
            # Obs: [pos(3), vel(3)=0, bat=1, state=0, target_delta(3), neighbors(9)=0]
            obs = np.concatenate([
                pos,
                np.zeros(3, dtype=np.float32),   # vel
                [1.0],                            # battery full
                [0.0],                            # state = ACTIVE
                target_delta,                     # target delta
                np.zeros(9, dtype=np.float32),    # no neighbors
            ])
            
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(model.device)
            with torch.no_grad():
                q_vals = model.q_net(obs_t).cpu().numpy().flatten()
            
            Q_max[j, i] = q_vals.max()
            Q_best_action[j, i] = q_vals.argmax()
    
    return xs, ys, Q_max, Q_best_action

target_demo = ep['target']
xs, ys, Q_max, Q_best = compute_qmap(model, env, target_demo, z_fixed=1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Q-value Heatmap tại Z=1.0m  |  Target: ({target_demo[0]:.2f}, {target_demo[1]:.2f}, {target_demo[2]:.2f})',
             fontsize=13, fontweight='bold')

# ── (1) Max Q-value heatmap ───────────────────────────────────
ax = axes[0]
im = ax.contourf(xs, ys, Q_max, levels=20, cmap='RdYlGn')
plt.colorbar(im, ax=ax, label='Max Q-value')
ax.scatter(target_demo[0], target_demo[1], marker='*', s=300, color='gold',
           edgecolors='k', zorder=5, label='Target')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Max Q-value (màu xanh = tốt)')
ax.legend()

# ── (2) Best action map ───────────────────────────────────────
ax = axes[1]
from matplotlib.colors import ListedColormap
cmap7 = ListedColormap(ACTION_COLORS)
im2 = ax.pcolormesh(xs, ys, Q_best, cmap=cmap7, vmin=-0.5, vmax=6.5)
cbar2 = plt.colorbar(im2, ax=ax, ticks=range(7))
cbar2.set_ticklabels([f'{i}: {n}' for i, n in enumerate(ACTION_NAMES)])
ax.scatter(target_demo[0], target_demo[1], marker='*', s=300, color='gold',
           edgecolors='k', zorder=5, label='Target')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Best Action tại mỗi điểm trong không gian')
ax.legend()

plt.tight_layout()
plt.savefig('plot_qvalue_heatmap.png', bbox_inches='tight')
plt.show()

print('💡 Diễn giải: Vùng màu xanh đậm = model tự tin cao. Màu đỏ = vùng khó/không chắc.')

## 8. Biểu đồ 5 — Multi-episode statistics

In [ ]:
print(f'Đang chạy {N_EVAL_EPISODES} episodes...')

all_episodes = []
for ep_idx in range(N_EVAL_EPISODES):
    ep_data = run_episode(env, model, seed=ep_idx, deterministic=True)
    all_episodes.append(ep_data)
    status = '✅' if ep_data['reached'] else '❌'
    print(f'  Ep {ep_idx+1:2d}: {status}  steps={ep_data["n_steps"]:3d}  '
          f'reward={ep_data["total_reward"]:7.2f}  '
          f'final_dist={ep_data["dists"][-1]:.3f}m')

# ── Tổng kết ──────────────────────────────────────────────────
success_mask   = np.array([e['reached']      for e in all_episodes])
all_rewards    = np.array([e['total_reward'] for e in all_episodes])
all_steps      = np.array([e['n_steps']      for e in all_episodes])
all_final_dist = np.array([e['dists'][-1]    for e in all_episodes])
steps_success  = all_steps[success_mask]

print(f'\n{'='*50}')
print(f'  Success rate       : {success_mask.mean()*100:.1f}% ({success_mask.sum()}/{N_EVAL_EPISODES})')
print(f'  Avg total reward   : {all_rewards.mean():.2f} ± {all_rewards.std():.2f}')
print(f'  Avg steps (all)    : {all_steps.mean():.1f} ± {all_steps.std():.1f}')
if steps_success.size > 0:
    print(f'  Avg steps (success): {steps_success.mean():.1f} ± {steps_success.std():.1f}')
print(f'  Avg final dist     : {all_final_dist.mean():.3f} ± {all_final_dist.std():.3f}m')
print(f'{'='*50}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'Multi-Episode Summary ({N_EVAL_EPISODES} episodes)', fontsize=14, fontweight='bold')

ep_ids = np.arange(1, N_EVAL_EPISODES + 1)
colors_ep = ['#27ae60' if s else '#e74c3c' for s in success_mask]

# ── (1) Total reward per episode ──────────────────────────────
ax = axes[0, 0]
bars = ax.bar(ep_ids, all_rewards, color=colors_ep, edgecolor='white')
ax.axhline(all_rewards.mean(), color='k', linestyle='--', linewidth=1.5,
           label=f'Mean: {all_rewards.mean():.1f}')
ax.set_title('Total Reward per Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── (2) Steps per episode ─────────────────────────────────────
ax = axes[0, 1]
ax.bar(ep_ids, all_steps, color=colors_ep, edgecolor='white')
ax.axhline(all_steps.mean(), color='k', linestyle='--', linewidth=1.5,
           label=f'Mean: {all_steps.mean():.1f}')
ax.set_title('Steps per Episode')
ax.set_xlabel('Episode'); ax.set_ylabel('Số steps')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── (3) Final distance per episode ───────────────────────────
ax = axes[0, 2]
ax.bar(ep_ids, all_final_dist, color=colors_ep, edgecolor='white')
ax.axhline(REACH_THRESHOLD, color='gold', linestyle='--', linewidth=1.5,
           label=f'Reach threshold ({REACH_THRESHOLD}m)')
ax.set_title('Final Distance to Goal')
ax.set_xlabel('Episode'); ax.set_ylabel('Khoảng cách (m)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── (4) Reward distribution ───────────────────────────────────
ax = axes[1, 0]
ax.hist(all_rewards[success_mask],  bins=8, alpha=0.7, color='#27ae60', label='Success')
ax.hist(all_rewards[~success_mask], bins=8, alpha=0.7, color='#e74c3c', label='Fail')
ax.set_title('Phân phối Total Reward')
ax.set_xlabel('Reward'); ax.set_ylabel('Số episode')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── (5) Distance curve — all episodes overlay ─────────────────
ax = axes[1, 1]
for i, ep_d in enumerate(all_episodes):
    color = '#27ae60' if ep_d['reached'] else '#e74c3c'
    ax.plot(ep_d['dists'], color=color, alpha=0.35, linewidth=1)
ax.axhline(REACH_THRESHOLD, color='gold', linestyle='--', linewidth=2,
           label=f'Reach threshold')
# Mean curve
max_len = max(len(e['dists']) for e in all_episodes)
padded  = [np.pad(e['dists'], (0, max_len - len(e['dists'])),
                  constant_values=e['dists'][-1]) for e in all_episodes]
mean_dist = np.mean(padded, axis=0)
ax.plot(mean_dist, color='k', linewidth=2.5, label='Mean', zorder=5)
ax.set_title('Distance to Goal — tất cả episodes')
ax.set_xlabel('Step'); ax.set_ylabel('Distance (m)')
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0],color='#27ae60',label='Success',lw=2),
              Line2D([0],[0],color='#e74c3c',label='Fail',lw=2),
              Line2D([0],[0],color='k',label='Mean',lw=2.5)]
ax.legend(handles=legend_els, fontsize=9); ax.grid(alpha=0.3)

# ── (6) Action distribution — tổng hợp ───────────────────────
ax = axes[1, 2]
all_actions = np.concatenate([e['actions'] for e in all_episodes])
counts = np.bincount(all_actions, minlength=7)
wedges, texts, autotexts = ax.pie(
    counts, labels=ACTION_NAMES, colors=ACTION_COLORS,
    autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
ax.set_title(f'Action Distribution tổng ({len(all_actions)} steps)')

# Legend cho success/fail màu
for ax_ in axes[0]:
    from matplotlib.patches import Patch
    legend_els2 = [Patch(facecolor='#27ae60', label='Success'),
                   Patch(facecolor='#e74c3c', label='Fail')]
    if ax_ == axes[0, 0]:
        ax_.legend(handles=[legend_els2[0], legend_els2[1],
                             ax_.get_legend().legend_handles[0]], fontsize=8)

plt.tight_layout()
plt.savefig('plot_multi_episode.png', bbox_inches='tight')
plt.show()

## 9. Biểu đồ 6 — So sánh Deterministic vs Stochastic policy

In [ ]:
# Chạy cùng 5 episode với policy deterministic vs stochastic (exploration)
N_COMPARE = 10
det_rewards, sto_rewards = [], []
det_success, sto_success = [], []
det_dist,    sto_dist    = [], []

for seed_i in range(N_COMPARE):
    det = run_episode(env, model, seed=seed_i, deterministic=True)
    sto = run_episode(env, model, seed=seed_i, deterministic=False)
    det_rewards.append(det['total_reward']); det_success.append(det['reached']); det_dist.append(det['dists'][-1])
    sto_rewards.append(sto['total_reward']); sto_success.append(sto['reached']); sto_dist.append(sto['dists'][-1])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Deterministic vs Stochastic Policy', fontsize=13, fontweight='bold')

x = np.arange(N_COMPARE)
w = 0.38

ax = axes[0]
ax.bar(x - w/2, det_rewards, width=w, label='Deterministic', color='#3498db', alpha=0.8)
ax.bar(x + w/2, sto_rewards, width=w, label='Stochastic',   color='#e67e22', alpha=0.8)
ax.set_title('Total Reward'); ax.set_xlabel('Episode'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(['Det', 'Sto'],
       [np.mean(det_success)*100, np.mean(sto_success)*100],
       color=['#3498db','#e67e22'], edgecolor='white', width=0.5)
ax.set_ylim(0, 110); ax.set_ylabel('%'); ax.set_title('Success Rate (%)')
for bar, val in zip(ax.patches, [np.mean(det_success)*100, np.mean(sto_success)*100]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.0f}%', ha='center', fontsize=12)
ax.grid(axis='y', alpha=0.3)

ax = axes[2]
ax.bar(x - w/2, det_dist, width=w, label='Deterministic', color='#3498db', alpha=0.8)
ax.bar(x + w/2, sto_dist, width=w, label='Stochastic',   color='#e67e22', alpha=0.8)
ax.axhline(REACH_THRESHOLD, color='gold', linestyle='--', linewidth=1.5, label='Reach threshold')
ax.set_title('Final Distance (m)'); ax.set_xlabel('Episode'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot_det_vs_sto.png', bbox_inches='tight')
plt.show()

## 10. Biểu đồ 7 — Trajectory XY & XZ overhead view

In [ ]:
# Chạy 5 episode với target cố định để so sánh đường đi
FIXED_TARGET = [3.0, 1.0, 1.5]
n_traj = 6
trajs = [run_episode(env, model, seed=i, fixed_target=FIXED_TARGET) for i in range(n_traj)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Trajectories với target cố định {FIXED_TARGET}  ({n_traj} runs)', fontsize=13, fontweight='bold')

traj_colors = plt.cm.tab10(np.linspace(0, 1, n_traj))

for ax_idx, (dim1, dim2, xlabel, ylabel, title) in enumerate([
    (0, 1, 'X (m)', 'Y (m)', 'Top view (XY)'),
    (0, 2, 'X (m)', 'Z (m)', 'Side view (XZ)'),
]):
    ax = axes[ax_idx]
    for i, traj in enumerate(trajs):
        pos = traj['positions']
        label = f'Run {i+1} {"✅" if traj["reached"] else "❌"}'
        ax.plot(pos[:, dim1], pos[:, dim2],
                color=traj_colors[i], alpha=0.75, linewidth=1.5, label=label)
        ax.scatter(pos[0, dim1], pos[0, dim2], s=60, color=traj_colors[i],
                   marker='o', edgecolors='k', zorder=4)
    
    ax.scatter(FIXED_TARGET[dim1], FIXED_TARGET[dim2],
               s=250, marker='*', color='gold', edgecolors='k', zorder=5, label='Target')
    
    # Vòng tròn reach threshold
    circle = plt.Circle((FIXED_TARGET[dim1], FIXED_TARGET[dim2]),
                         REACH_THRESHOLD, color='gold', fill=False,
                         linestyle='--', linewidth=1.5, alpha=0.7)
    ax.add_patch(circle)
    
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)
    ax.set_aspect('equal', adjustable='datalim')

plt.tight_layout()
plt.savefig('plot_multi_traj.png', bbox_inches='tight')
plt.show()

## 11. Chẩn đoán — Các dấu hiệu cần chú ý

In [ ]:
print('=' * 60)
print('  CHẨN ĐOÁN MODEL')
print('=' * 60)

sr = success_mask.mean() * 100
avg_r = all_rewards.mean()
idle_frac = (np.concatenate([e['actions'] for e in all_episodes]) == 6).mean() * 100
avg_final = all_final_dist.mean()

checks = [
    (sr >= 70,  f'Success rate {sr:.1f}%',
     '✅ Tốt (>70%)'    if sr >= 70 else
     '⚠️  Trung bình'  if sr >= 40 else
     '❌ Cần train thêm hoặc check reward'),

    (avg_r > 0, f'Avg reward {avg_r:.2f}',
     '✅ Dương — model đang học đúng hướng' if avg_r > 0 else
     '❌ Âm — reward shaping cần xem lại'),

    (idle_frac < 20, f'"Đứng im" chiếm {idle_frac:.1f}%',
     '✅ Bình thường' if idle_frac < 20 else
     '⚠️  Cao — model có thể bị lazy policy (luôn đứng im để tránh penalty)'),

    (avg_final < 1.0, f'Avg final dist {avg_final:.3f}m',
     '✅ Gần đích' if avg_final < 0.5 else
     '⚠️  Còn xa — train thêm hoặc tăng W_REACH' if avg_final < 1.5 else
     '❌ Xa đích nhiều — cần kiểm tra action space / physics'),
]

for passed, metric, msg in checks:
    print(f'  {metric:<35} → {msg}')

print()
print('  HƯỚNG DẪN ĐỌC BIỂU ĐỒ:')
print('  • Trajectory 3D  : đường ngoằn ngoèo quá → action space quá nhỏ (STEP_SIZE nhỏ)')
print('  • Q-value heatmap: nếu không có gradient rõ → model chưa học được spatial reasoning')
print('  • Action dist    : nếu 1 action chiếm >60% → policy collapse hoặc reward bias')
print('  • Det vs Sto     : nếu stochastic tốt hơn det → model còn underfit, cần train thêm')
print('=' * 60)

In [ ]:
env.close()
print('✅ Env đã đóng. Tất cả biểu đồ đã được lưu vào thư mục hiện tại.')